# Nova Workshop 2026
<br/>
<img src="https://www.polyestertime.com/wp-content/uploads/2017/01/Nova-Chemical-23-09-2016.jpg" />
<br/><br/>

## Introduction
In this module, we're going to start working with Lakebase which is a transactional, relational-style interface. Essentially it's an abstraction of PostgreSQL that runs on dedicated resources within Databricks and offers delta capabilities with OLTP behaviour.

- A database instance has been set up for this workshop

### Goal:
- Create a per-user schema in Lakebase.
- Create an `incidents` table per user.
- Seed at least one incident from your Gold KPI table.

In [0]:
user_schema = "andrij_demo"        # TODO
catalog_name = "nova_workshop"
lakebase_db = "nova-incidents"     # The Lakebase database name

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {user_schema}")

gold_table = f"{catalog_name}.{user_schema}.gold_daily_line_kpis"
incidents_table = f"{catalog_name}.{user_schema}.gold_incidents"

print("Gold KPIs table   :", gold_table)
print("Incidents table   :", incidents_table)

## Step 1 – Create Schema &amp; Incidents Table in Lakebase
First we will create a new table that tracks just our incidents. This will be done in Lakehouse - typically in some type of process pipeline. For this workshop, we will be manually updating the table with an incident. This table will be mapped to a Lakebase instance for access via a Databricks App. Since the alert flag was set up on a daily basis, for any day that exceeds 3.25% error rate we should have a relatively high rate of incidence for alert events.

In [0]:
# Create the incidents table if it does not exist, with schema and Delta format
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {incidents_table} (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY,
  line_id STRING NOT NULL,
  ts TIMESTAMP NOT NULL,
  severity STRING,
  summary STRING,
  details STRING,
  status STRING
)
USING DELTA
""")

## Step 2 – Seed an Incident from a Gold Alert

We take our alerts with `alert = true` and create a synthetic set of HIGH-severity incidents.

In [0]:
# Insert new incident records for lines with alerts from the gold KPIs table
spark.sql(f"""
INSERT INTO {incidents_table} (line_id, ts, severity, summary, details, status)
SELECT
  line_id,
  day,
  'HIGH',
  CONCAT('High BAD rate on ', CAST(day AS STRING)),
  'Auto-generated from Gold KPIs.',
  'OPEN'
FROM {gold_table}
WHERE alert = TRUE;
""")

## Step 3 – Quick Check

In [0]:
# Display the incidents table containing records of line incidents
display(spark.table(incidents_table))

## Step 4 - Create a synchronized table with Lakebase

To create a synchronized table with a Lakebase instance:

1. **Create your source table in the Lakehouse**  
   Prepare the Delta table you want to sync (e.g., your `incidents` table).

2. **Map the table to your Lakebase instance**  
   Use Databricks SQL or the UI to connect your Lakebase instance and map the table.

3. **Enable synchronization**  
   Use the `CREATE TABLE ... SYNC TO LAKEBASE` command or the Databricks UI to enable sync.  
   Example SQL:
   sql
   CREATE TABLE lakebase_db.incidents
   SYNC TO LAKEBASE
   AS SELECT * FROM lakehouse_db.incidents;
   

4. **Verify synchronization**  
   Query the table from your Lakebase instance to confirm data is available and up-to-date.

> This process ensures your Delta table is available for OLTP workloads via Lakebase with transactional consistency.